# Aula 5 — DataFrames com Pandas
**Introdução à Programação para Pesquisa Biomédica · IBCCF/UFRJ**

In [ ]:
import pandas as pd
import numpy as np

# Criar dataset de exemplo
np.random.seed(42)
n = 60
df = pd.DataFrame({
    "gene":       [f"GENE{i:03d}" for i in range(n)],
    "organismo":  np.random.choice(["Homo sapiens","Mus musculus","Danio rerio"], n),
    "condicao":   np.random.choice(["controle","tratamento"], n),
    "expressao":  np.round(np.random.lognormal(1.5, 0.8, n), 3),
    "gc_pct":     np.round(np.random.uniform(35, 70, n), 1),
})
print(df.shape)
df.head()

## 1. Inspecionar o DataFrame

In [ ]:
print(df.dtypes)
print()
df.describe()

## 2. Selecionar colunas e filtrar linhas

In [ ]:
# Selecionar colunas
df[["gene", "expressao"]].head()

# Filtrar
humanos = df[df["organismo"] == "Homo sapiens"]
print(f"Amostras humanas: {len(humanos)}")

# Múltiplas condições
filtrado = df[(df["expressao"] > 5) & (df["condicao"] == "tratamento")]
print(f"Tratamento + alta expressão: {len(filtrado)}")

## 3. Adicionar colunas

In [ ]:
df["log2_expr"] = np.log2(df["expressao"])
df["grupo"] = df["expressao"].apply(lambda x: "alto" if x > 5 else "baixo")
df.head()

## 4. Groupby e sumarização

In [ ]:
resumo = df.groupby("organismo").agg(
    media_expr  = ("expressao", "mean"),
    dp_expr     = ("expressao", "std"),
    n_amostras  = ("gene", "count")
).round(3).reset_index()
print(resumo)

## 5. Merge / join

In [ ]:
# Criar segundo DataFrame com metadados
meta = pd.DataFrame({
    "organismo": ["Homo sapiens", "Mus musculus", "Danio rerio"],
    "nome_comum": ["Humano", "Camundongo", "Zebrafish"],
    "genoma_Mb": [3200, 2700, 1400]
})

df_merged = pd.merge(df, meta, on="organismo")
df_merged[["gene","organismo","nome_comum","genoma_Mb","expressao"]].head()

## 6. Dados faltantes

In [ ]:
# Introduzir NAs artificialmente
df_na = df.copy()
df_na.loc[np.random.choice(df.index, 8, replace=False), "expressao"] = np.nan

print("NAs por coluna:")
print(df_na.isna().sum())

# Remover NAs
df_clean = df_na.dropna(subset=["expressao"])
print(f"\nLinhas antes: {len(df_na)} | depois: {len(df_clean)}")

## Exercício
Filtre as amostras de *Homo sapiens* em condição de tratamento, calcule o percentil 75 de expressão e identifique os genes acima desse valor.

In [ ]:
# Sua solução:
humanos_trat = df[(df["organismo"]=="Homo sapiens") & (df["condicao"]=="tratamento")]
p75 = humanos_trat["expressao"].quantile(0.75)
genes_top = humanos_trat[humanos_trat["expressao"] > p75]["gene"].tolist()
print(f"Percentil 75: {p75:.3f}")
print(f"Genes acima: {genes_top[:5]}..." if len(genes_top)>5 else f"Genes: {genes_top}")